In [7]:

import torch
import random
import numpy as np

# Set random seed for reproducibility
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

# Choose GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"

print("PyTorch version:", torch.__version__)
print("Device:", device)

if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


PyTorch version: 2.11.0+cu130
Device: cuda
GPU: Tesla T4


In [8]:

import urllib.request

# Download Tiny Shakespeare dataset
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"

urllib.request.urlretrieve(url, "input.txt")

# Read the dataset
with open("input.txt", "r", encoding="utf-8") as f:
    text = f.read()

# Check dataset
print("Total characters:", len(text))
print("Unique characters:", len(set(text)))

# Show the first 400 characters
print("\nSample text:")
print(text[:400])


Total characters: 1115394
Unique characters: 65

Sample text:
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it 


In [9]:

# Get all unique characters
chars = sorted(list(set(text)))
vocab_size = len(chars)

# Create character-to-integer mappings
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

# Encode text into integers
def encode(s):
    return [stoi[c] for c in s]

# Decode integers back into text
def decode(numbers):
    return ''.join([itos[i] for i in numbers])

print("Vocabulary size:", vocab_size)

# Test tokenizer
sample = "hello"
encoded = encode(sample)

print("Original:", sample)
print("Encoded:", encoded)
print("Decoded:", decode(encoded))


Vocabulary size: 65
Original: hello
Encoded: [46, 43, 50, 50, 53]
Decoded: hello


In [10]:

# Convert the entire text into integers
data = torch.tensor(encode(text), dtype=torch.long)

# 90% training, 10% validation
n = int(0.9 * len(data))

train_data = data[:n]
val_data = data[n:]

print("Total tokens:", len(data))
print("Training tokens:", len(train_data))
print("Validation tokens:", len(val_data))


Total tokens: 1115394
Training tokens: 1003854
Validation tokens: 111540


In [11]:

# Training settings
batch_size = 32
block_size = 64

def get_batch(split):
    # Choose dataset
    data_source = train_data if split == "train" else val_data

    # Random starting positions
    ix = torch.randint(
        len(data_source) - block_size,
        (batch_size,)
    )

    # Input sequences
    x = torch.stack([
        data_source[i:i + block_size] for i in ix
    ])

    # Target sequences (shifted by one character)
    y = torch.stack([
        data_source[i + 1:i + block_size + 1] for i in ix
    ])

    return x.to(device), y.to(device)

# Test batch generator
x, y = get_batch("train")

print("Input shape:", x.shape)
print("Target shape:", y.shape)

print("\nInput example:")
print(decode(x[0].tolist()))

print("\nTarget example:")
print(decode(y[0].tolist()))


Input shape: torch.Size([32, 64])
Target shape: torch.Size([32, 64])

Input example:
Paris!
What said my man, when my betossed soul
Did not attend hi

Target example:
aris!
What said my man, when my betossed soul
Did not attend him


**Bigram Model**

In [12]:

import torch
import torch.nn as nn
import torch.nn.functional as F

class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()

        # Predict next character using current character
        self.token_embedding_table = nn.Embedding(
            vocab_size, vocab_size
        )

    def forward(self, idx, targets=None):

        # Shape: (batch_size, block_size, vocab_size)
        logits = self.token_embedding_table(idx)

        loss = None

        if targets is not None:
            B, T, C = logits.shape

            logits_flat = logits.reshape(B * T, C)
            targets_flat = targets.reshape(B * T)

            loss = F.cross_entropy(
                logits_flat, targets_flat
            )

        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):

        for _ in range(max_new_tokens):

            # Bigram only needs the last character
            logits, _ = self(idx[:, -1:])

            # Get next-character probabilities
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)

            # Sample the next character
            next_idx = torch.multinomial(probs, num_samples=1)

            # Append it to the sequence
            idx = torch.cat((idx, next_idx), dim=1)

        return idx


# Initialize model
bigram_model = BigramLanguageModel(vocab_size).to(device)

print("Bigram model created!")
print("Parameters:", sum(
    p.numel() for p in bigram_model.parameters()
))


Bigram model created!
Parameters: 4225


In [13]:

@torch.no_grad()
def estimate_loss(model, split, eval_batches=20):

    model.eval()

    data_source = train_data if split == "train" else val_data

    # Same evaluation samples for all three models
    seed = 1234 if split == "train" else 5678
    generator = torch.Generator().manual_seed(seed)

    losses = []

    for _ in range(eval_batches):

        ix = torch.randint(
            len(data_source) - block_size,
            (batch_size,),
            generator=generator
        )

        x = torch.stack([
            data_source[i:i + block_size] for i in ix
        ]).to(device)

        y = torch.stack([
            data_source[i + 1:i + block_size + 1]
            for i in ix
        ]).to(device)

        logits, loss = model(x, y)
        losses.append(loss.item())

    model.train()

    return sum(losses) / len(losses)


In [14]:

import time

# Training settings
max_steps = 1000
learning_rate = 0.01
eval_interval = 250

optimizer = torch.optim.AdamW(
    bigram_model.parameters(),
    lr=learning_rate
)

start_time = time.perf_counter()

# Training loop
for step in range(max_steps):

    # Get training batch
    xb, yb = get_batch("train")

    # Forward pass
    logits, loss = bigram_model(xb, yb)

    # Backpropagation
    optimizer.zero_grad(set_to_none=True)
    loss.backward()

    # Update model parameters
    optimizer.step()

    # Evaluate every 250 steps
    if (step + 1) % eval_interval == 0:

        train_loss = estimate_loss(
            bigram_model, "train"
        )

        val_loss = estimate_loss(
            bigram_model, "val"
        )

        print(
            f"Step {step + 1}: "
            f"Train Loss = {train_loss:.4f}, "
            f"Val Loss = {val_loss:.4f}"
        )

# Record total time
bigram_time = time.perf_counter() - start_time

# Final evaluation
bigram_train_loss = estimate_loss(bigram_model, "train")
bigram_val_loss = estimate_loss(bigram_model, "val")

print("\nTraining complete!")
print(f"Training Loss: {bigram_train_loss:.4f}")
print(f"Validation Loss: {bigram_val_loss:.4f}")
print(f"Training Time: {bigram_time:.2f} seconds")


Step 250: Train Loss = 2.7781, Val Loss = 2.7937
Step 500: Train Loss = 2.5315, Val Loss = 2.5490
Step 750: Train Loss = 2.4922, Val Loss = 2.5163
Step 1000: Train Loss = 2.4796, Val Loss = 2.5047

Training complete!
Training Loss: 2.4796
Validation Loss: 2.5047
Training Time: 2.71 seconds


In [15]:

# Start with a newline character
start = torch.tensor(
    [[stoi["\n"]]],
    dtype=torch.long,
    device=device
)

bigram_model.eval()

# Generate 300 new characters
torch.manual_seed(42)
generated = bigram_model.generate(
    start,
    max_new_tokens=300
)

bigram_text = decode(generated[0].tolist())

print("Bigram Generated Text:")
print("----------------------")
print(bigram_text)


Bigram Generated Text:
----------------------

I
O: asafite tis, te othur mod thand he, preckn,

Henthimavachishelapinisers we s, oreanotsthelu.
GLI it inde Miced tlat mang yofowhas
brothangean t.


D hathapr me machan fHe.
Ben d iere--sthurore cense wn ce me, woass t avest t, metha'?

Mavy?


TINGS:

D:BRGLONGof ind s heve she, fHavers ndowe:
L


In [16]:

# Store experiment results
results = {}

results["Bigram"] = {
    "train_loss": bigram_train_loss,
    "val_loss": bigram_val_loss,
    "training_time": bigram_time,
    "generated_text": bigram_text
}

print("Bigram results saved!")
print(results["Bigram"])


Bigram results saved!
{'train_loss': 2.4795971393585203, 'val_loss': 2.5046753168106077, 'training_time': 2.713375726000095, 'generated_text': "\nI\nO: asafite tis, te othur mod thand he, preckn,\n\nHenthimavachishelapinisers we s, oreanotsthelu.\nGLI it inde Miced tlat mang yofowhas\nbrothangean t.\n\n\nD hathapr me machan fHe.\nBen d iere--sthurore cense wn ce me, woass t avest t, metha'?\n\nMavy?\n\n\nTINGS:\n\nD:BRGLONGof ind s heve she, fHavers ndowe:\nL"}


**LSTM Model**

In [17]:

class LSTMLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, 64)

        self.lstm = nn.LSTM(
            input_size=64,
            hidden_size=128,
            num_layers=1,
            batch_first=True
        )

        self.fc = nn.Linear(128, vocab_size)

    def forward(self, idx, targets=None):

        # Convert tokens into embeddings
        x = self.embedding(idx)

        # Process sequence using LSTM
        x, hidden = self.lstm(x)

        # Predict next character
        logits = self.fc(x)

        loss = None

        if targets is not None:
            B, T, C = logits.shape

            logits_flat = logits.reshape(B * T, C)
            targets_flat = targets.reshape(B * T)

            loss = F.cross_entropy(
                logits_flat, targets_flat
            )

        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):

        self.eval()

        # Process initial prompt
        x = self.embedding(idx)
        x, hidden = self.lstm(x)

        for _ in range(max_new_tokens):

            logits = self.fc(x[:, -1, :])
            probs = F.softmax(logits, dim=-1)

            next_idx = torch.multinomial(
                probs, num_samples=1
            )

            idx = torch.cat((idx, next_idx), dim=1)

            # Reuse hidden state for the next character
            x = self.embedding(next_idx)
            x, hidden = self.lstm(x, hidden)

        return idx


# Create LSTM model
lstm_model = LSTMLanguageModel(vocab_size).to(device)

print("LSTM model created!")
print("Parameters:", sum(
    p.numel() for p in lstm_model.parameters()
))


LSTM model created!
Parameters: 111873


In [18]:

# LSTM training settings
lstm_steps = 1000
lstm_lr = 0.003
eval_interval = 250

optimizer = torch.optim.AdamW(
    lstm_model.parameters(),
    lr=lstm_lr
)

# Synchronize GPU before timing
if device == "cuda":
    torch.cuda.synchronize()

start_time = time.perf_counter()

for step in range(lstm_steps):

    xb, yb = get_batch("train")

    # Forward pass
    logits, loss = lstm_model(xb, yb)

    # Backpropagation
    optimizer.zero_grad(set_to_none=True)
    loss.backward()

    # Prevent exploding gradients
    torch.nn.utils.clip_grad_norm_(
        lstm_model.parameters(), 1.0
    )

    optimizer.step()

    if (step + 1) % eval_interval == 0:

        train_loss = estimate_loss(
            lstm_model, "train"
        )

        val_loss = estimate_loss(
            lstm_model, "val"
        )

        print(
            f"Step {step + 1}: "
            f"Train Loss = {train_loss:.4f}, "
            f"Val Loss = {val_loss:.4f}"
        )

# Synchronize GPU after training
if device == "cuda":
    torch.cuda.synchronize()

lstm_time = time.perf_counter() - start_time

lstm_train_loss = estimate_loss(lstm_model, "train")
lstm_val_loss = estimate_loss(lstm_model, "val")

print("\nLSTM Training Complete!")
print(f"Training Loss: {lstm_train_loss:.4f}")
print(f"Validation Loss: {lstm_val_loss:.4f}")
print(f"Training Time: {lstm_time:.2f} seconds")


Step 250: Train Loss = 1.9557, Val Loss = 2.0256
Step 500: Train Loss = 1.7767, Val Loss = 1.8977
Step 750: Train Loss = 1.6935, Val Loss = 1.8239
Step 1000: Train Loss = 1.6476, Val Loss = 1.7797

LSTM Training Complete!
Training Loss: 1.6476
Validation Loss: 1.7797
Training Time: 5.09 seconds


In [19]:

# Start with the same prompt as Bigram
start = torch.tensor(
    [[stoi["\n"]]],
    dtype=torch.long,
    device=device
)

torch.manual_seed(42)

generated = lstm_model.generate(
    start,
    max_new_tokens=300
)

lstm_text = decode(generated[0].tolist())

print("LSTM Generated Text:")
print("--------------------")
print(lstm_text)


LSTM Generated Text:
--------------------

If tile; noblen'd the till me is and he beteck.

GLOUCESTERY:
So, lapvits by we have good
The luke I will, be ever fold
He call it thy
brothange.
Secture his gream, my hand Hereeding in wife,
Whose cansely make that as nower? whither?

PRINCE:
Your I have are old God in Wadery: she, be bron than sta


In [20]:

results["LSTM"] = {
    "train_loss": lstm_train_loss,
    "val_loss": lstm_val_loss,
    "training_time": lstm_time,
    "generated_text": lstm_text
}

print("LSTM results saved!")

# Compare Bigram and LSTM
for name, result in results.items():
    print(f"\n{name}")
    print("Train Loss:", round(result["train_loss"], 4))
    print("Val Loss:", round(result["val_loss"], 4))
    print("Training Time:", round(result["training_time"], 2))


LSTM results saved!

Bigram
Train Loss: 2.4796
Val Loss: 2.5047
Training Time: 2.71

LSTM
Train Loss: 1.6476
Val Loss: 1.7797
Training Time: 5.09


**Transformer Model**

In [21]:

class TransformerBlock(nn.Module):

    def __init__(self, n_embd, n_head):
        super().__init__()

        # Self-attention layer
        self.attention = nn.MultiheadAttention(
            embed_dim=n_embd,
            num_heads=n_head,
            batch_first=True
        )

        # Feedforward network
        self.feedforward = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd)
        )

        # Layer normalization
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):

        T = x.shape[1]

        # Prevent attention to future characters
        mask = torch.triu(
            torch.ones(T, T, device=x.device),
            diagonal=1
        ).bool()

        # Self-attention
        x_norm = self.ln1(x)

        attn_output, _ = self.attention(
            x_norm,
            x_norm,
            x_norm,
            attn_mask=mask,
            need_weights=False
        )

        x = x + attn_output

        # Feedforward network
        x = x + self.feedforward(self.ln2(x))

        return x


In [22]:

class TransformerLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()

        n_embd = 64
        n_head = 4
        n_layer = 2

        # Token embeddings
        self.token_embedding = nn.Embedding(
            vocab_size, n_embd
        )

        # Positional embeddings
        self.position_embedding = nn.Embedding(
            block_size, n_embd
        )

        # Transformer blocks
        self.blocks = nn.Sequential(
            *[
                TransformerBlock(n_embd, n_head)
                for _ in range(n_layer)
            ]
        )

        # Final layers
        self.ln = nn.LayerNorm(n_embd)
        self.fc = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):

        B, T = idx.shape

        # Token embeddings
        token_emb = self.token_embedding(idx)

        # Positional embeddings
        positions = torch.arange(
            T, device=idx.device
        )
        pos_emb = self.position_embedding(positions)

        # Combine token and position information
        x = token_emb + pos_emb

        # Transformer layers
        x = self.blocks(x)
        x = self.ln(x)

        # Predict next character
        logits = self.fc(x)

        loss = None

        if targets is not None:
            B, T, C = logits.shape

            logits_flat = logits.reshape(B * T, C)
            targets_flat = targets.reshape(B * T)

            loss = F.cross_entropy(
                logits_flat, targets_flat
            )

        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens):

        self.eval()

        for _ in range(max_new_tokens):

            # Keep only the last block_size tokens
            idx_cond = idx[:, -block_size:]

            logits, _ = self(idx_cond)

            # Last position predicts next character
            logits = logits[:, -1, :]

            probs = F.softmax(logits, dim=-1)

            next_idx = torch.multinomial(
                probs, num_samples=1
            )

            idx = torch.cat(
                (idx, next_idx), dim=1
            )

        return idx


# Initialize Transformer
torch.manual_seed(42)

transformer_model = TransformerLanguageModel(
    vocab_size
).to(device)

print("Transformer model created!")
print("Parameters:", sum(
    p.numel() for p in transformer_model.parameters()
))


Transformer model created!
Parameters: 112577


In [23]:

# Transformer training settings
transformer_steps = 1000
transformer_lr = 0.001
eval_interval = 250

optimizer = torch.optim.AdamW(
    transformer_model.parameters(),
    lr=transformer_lr
)

if device == "cuda":
    torch.cuda.synchronize()

start_time = time.perf_counter()

for step in range(transformer_steps):

    xb, yb = get_batch("train")

    # Forward pass
    logits, loss = transformer_model(xb, yb)

    # Backpropagation
    optimizer.zero_grad(set_to_none=True)
    loss.backward()

    # Gradient clipping
    torch.nn.utils.clip_grad_norm_(
        transformer_model.parameters(), 1.0
    )

    optimizer.step()

    # Evaluate every 250 steps
    if (step + 1) % eval_interval == 0:

        train_loss = estimate_loss(
            transformer_model, "train"
        )

        val_loss = estimate_loss(
            transformer_model, "val"
        )

        print(
            f"Step {step + 1}: "
            f"Train Loss = {train_loss:.4f}, "
            f"Val Loss = {val_loss:.4f}"
        )

if device == "cuda":
    torch.cuda.synchronize()

transformer_time = time.perf_counter() - start_time

# Final evaluation
transformer_train_loss = estimate_loss(
    transformer_model, "train"
)

transformer_val_loss = estimate_loss(
    transformer_model, "val"
)

print("\nTransformer Training Complete!")
print(f"Training Loss: {transformer_train_loss:.4f}")
print(f"Validation Loss: {transformer_val_loss:.4f}")
print(f"Training Time: {transformer_time:.2f} seconds")


Step 250: Train Loss = 2.4635, Val Loss = 2.4684
Step 500: Train Loss = 2.2825, Val Loss = 2.3036
Step 750: Train Loss = 2.1479, Val Loss = 2.1782
Step 1000: Train Loss = 2.0498, Val Loss = 2.0961

Transformer Training Complete!
Training Loss: 2.0498
Validation Loss: 2.0961
Training Time: 8.12 seconds


In [24]:

# Same starting prompt
start = torch.tensor(
    [[stoi["\n"]]],
    dtype=torch.long,
    device=device
)

torch.manual_seed(42)

generated = transformer_model.generate(
    start,
    max_new_tokens=300
)

transformer_text = decode(
    generated[0].tolist()
)

print("Transformer Generated Text:")
print("---------------------------")
print(transformer_text)


Transformer Generated Text:
---------------------------

Prit Cowfiter, 'ther on up monstarn on eep:
Ton,
Of I witho--ASThelleveis by we burorn.

PUANCETRENIICENT,NG HABER:
Plakeman, your thy brothangsan try
Thas thaps me machan the.

Nuplice:
Histwurs:
Icense wilce my seyess thee? wor, ment;
Alm; hy in that hy ass ole!-
In fir dervals:

TENVIO:
Ky and:
T


In [25]:

import pandas as pd

# Save Transformer results
results["Transformer"] = {
    "train_loss": transformer_train_loss,
    "val_loss": transformer_val_loss,
    "training_time": transformer_time,
    "generated_text": transformer_text
}

# Create comparison table
comparison = []

for name, result in results.items():

    comparison.append({
        "Model": name,
        "Train Loss": round(result["train_loss"], 4),
        "Validation Loss": round(result["val_loss"], 4),
        "Training Time (s)": round(result["training_time"], 2)
    })

df = pd.DataFrame(comparison)

display(df)

# Save results as CSV
df.to_csv("model_comparison.csv", index=False)

print("Comparison results saved!")


,Model,Train Loss,Validation Loss,Training Time (s)
0,Bigram,2.4796,2.5047,2.71
1,LSTM,1.6476,1.7797,5.09
2,Transformer,2.0498,2.0961,8.12


Comparison results saved!


**Evaluation**

In [26]:

# Compare generated text from all three models

for name in ["Bigram", "LSTM", "Transformer"]:

    print("=" * 50)
    print(name, "Generated Text")
    print("=" * 50)

    print(results[name]["generated_text"])
    print()


Bigram Generated Text

I
O: asafite tis, te othur mod thand he, preckn,

Henthimavachishelapinisers we s, oreanotsthelu.
GLI it inde Miced tlat mang yofowhas
brothangean t.


D hathapr me machan fHe.
Ben d iere--sthurore cense wn ce me, woass t avest t, metha'?

Mavy?


TINGS:

D:BRGLONGof ind s heve she, fHavers ndowe:
L

LSTM Generated Text

If tile; noblen'd the till me is and he beteck.

GLOUCESTERY:
So, lapvits by we have good
The luke I will, be ever fold
He call it thy
brothange.
Secture his gream, my hand Hereeding in wife,
Whose cansely make that as nower? whither?

PRINCE:
Your I have are old God in Wadery: she, be bron than sta

Transformer Generated Text

Prit Cowfiter, 'ther on up monstarn on eep:
Ton,
Of I witho--ASThelleveis by we burorn.

PUANCETRENIICENT,NG HABER:
Plakeman, your thy brothangsan try
Thas thaps me machan the.

Nuplice:
Histwurs:
Icense wilce my seyess thee? wor, ment;
Alm; hy in that hy ass ole!-
In fir dervals:

TENVIO:
Ky and:
T



In [27]:

# Save all generated text samples

with open("generated_samples.txt", "w", encoding="utf-8") as f:

    for name in ["Bigram", "LSTM", "Transformer"]:

        f.write("=" * 50 + "\n")
        f.write(name + " Generated Text\n")
        f.write("=" * 50 + "\n")

        f.write(results[name]["generated_text"])
        f.write("\n\n")

print("Generated samples saved!")


Generated samples saved!


## Results and Analysis

I trained three character-level language models on the Tiny Shakespeare dataset: Bigram, LSTM, and a small Transformer. All three models used the same dataset, tokenizer, training and validation split, batch size, context window, and 1,000 training steps.

### 1. Model Performance Comparison

| Model | Training Loss | Validation Loss | Training Time |
|---|---:|---:|---:|
| Bigram | 2.4796 | 2.5047 | 2.71s |
| LSTM | 1.6476 | 1.7797 | 5.09s |
| Transformer | 2.0498 | 2.0961 | 8.12s |

### 2. Which Model Performed Best?

The LSTM achieved the lowest validation loss of 1.7797, compared to 2.5047 for Bigram and 2.0961 for Transformer. Its validation loss was approximately 28.9% lower than Bigram's.

This suggests that LSTM was the most effective at predicting the next character under my experimental settings. Unlike Bigram, which only considers the current character, LSTM can use information from previous characters through its hidden states.

### 3. Which Model Trained Fastest?

The Bigram model had the shortest recorded training time at 2.71 seconds, followed by LSTM at 5.09 seconds and Transformer at 8.12 seconds.

This makes sense because Bigram has a much simpler architecture. LSTM requires recurrent computations, while Transformer performs multiple attention and feedforward operations.

### 4. Which Model Generated the Most Realistic Text?

The LSTM generated the most realistic text in my experiment.

The Bigram model mostly produced random-looking combinations of characters, such as "Henthimavachishelapinisers," showing that it struggled to form meaningful words and sentences.

The LSTM generated more recognizable English words and Shakespeare-style dialogue structures, including "PRINCE:" and phrases such as "I will, be ever fold." Although the sentences were not completely grammatically correct, they were more readable than the Bigram output.

The Transformer also learned some Shakespeare-style formatting, including character names, punctuation, and dialogue. However, it still produced many incomplete or invented words, making its output less coherent than the LSTM's.

### 5. What Did the Transformer Improve?

The Transformer achieved a lower validation loss than Bigram by using self-attention to incorporate information from multiple previous characters.

Self-attention allows each position to directly consider relevant earlier positions in the sequence. This is different from Bigram, which only uses one character, and LSTM, which processes information through recurrent hidden states.

However, the Transformer did not outperform LSTM in my experiment. This may be related to the limited number of training steps, the small Transformer architecture, or differences in hyperparameters.

### 6. Was the Added Complexity Worth It?

Under my experimental settings, LSTM provided the best balance between prediction performance and training time.

The Transformer required more training time but achieved a higher validation loss than LSTM. Therefore, its added complexity did not provide a performance advantage in this particular experiment.

However, Transformers remain powerful because self-attention can model relationships between distant positions and allows more parallel computation during training.

### 7. Experimental Limitations

Although all three models used the same dataset and number of training steps, they had different architectures, parameter counts, and learning rates. Therefore, this experiment was not a comparison under identical computational budgets.

The recorded training times are approximate, and GPU timing methods were not completely consistent across the initial experiments.

Additionally, text quality was evaluated qualitatively using a small number of generated samples. More training steps, additional hyperparameter tuning, and multiple random seeds could provide a more reliable comparison.

---

## Final Reflection

After comparing Bigram, LSTM, and Transformer models, I learned that using more context can significantly improve language modeling.

Bigram only considers the current character, which limits its ability to learn meaningful words and sentence structures. LSTM improves on this by using hidden states to remember information from previous characters. Transformers take a different approach by using self-attention to directly examine relationships between positions in the input sequence.

I think self-attention is what makes Transformers especially powerful for language modeling. It allows the model to identify relevant information from different parts of the context without relying entirely on sequential hidden-state updates. It also supports parallel processing during training, which becomes important for larger models and datasets.

Although my Transformer did not achieve the best results, this experiment showed me that a more advanced architecture does not automatically guarantee better performance. Model size, training time, hyperparameters, and available computational resources all influence the final results.

Overall, this project helped me better understand the differences between traditional statistical models, recurrent neural networks, and attention-based models.

In [28]:

import os

# Check required result files
files_to_check = [
    "model_comparison.csv",
    "generated_samples.txt"
]

for filename in files_to_check:
    if os.path.exists(filename):
        print(filename, "- Found")
    else:
        print(filename, "- Missing")


model_comparison.csv - Found
generated_samples.txt - Found
